# Vela Compiler for TFLite Models

This notebook demonstrates how to compile TFLite models using Ethos-U Vela compiler for optimization on ARM Ethos-U NPUs.

## 1. Install Vela Compiler

First, we need to install the Ethos-U Vela compiler.

In [1]:
%pip install ethos-u-vela

  Using cached flatbuffers-24.3.25-py2.py3-none-any.whl.metadata (850 bytes)
Using cached flatbuffers-24.3.25-py2.py3-none-any.whl (26 kB)
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19
Note: you may need to restart the kernel to use updated packages.


The system cannot find the path specified.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.9 requires protobuf<4,>=3.11, but you have protobuf 6.33.5 which is incompatible.
tensorflow-intel 2.13.0 requires keras<2.14,>=2.13.1, but you have keras 3.13.2 which is incompatible.
tensorflow-intel 2.13.0 requires numpy<=1.24.3,>=1.22, but you have numpy 2.4.1 which is incompatible.
tensorflow-intel 2.13.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.
tensorflow-intel 2.13.0 requires tensorboard<2.14,>=2.13, but you have tensorboard 2.20.0 which is incompatible.
tensorflow-intel 2.13.0 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-extensions 4.15.0 which is incompatible.


## 2. Import Required Libraries

In [2]:
import os
import subprocess
from pathlib import Path

## 3. Define Vela Compilation Function

This function will compile a TFLite model and save it with the `_vela` suffix.

In [3]:
def compile_with_vela(input_tflite_path, output_dir=None, config_file=None, accelerator_config="ethos-u55-128"):
    """
    Compile a TFLite model using Vela compiler.
    
    Parameters:
    -----------
    input_tflite_path : str
        Path to the input TFLite model file
    output_dir : str, optional
        Directory to save the compiled model. If None, uses the same directory as input
    config_file : str, optional
        Path to Vela configuration file
    accelerator_config : str, default="ethos-u55-128"
        Accelerator configuration (e.g., "ethos-u55-128", "ethos-u55-256", "ethos-u65-256")
    
    Returns:
    --------
    str : Path to the compiled TFLite model
    """
    # Convert to Path object
    input_path = Path(input_tflite_path)
    
    # Check if input file exists
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_tflite_path}")
    
    # Determine output directory
    if output_dir is None:
        output_dir = input_path.parent
    else:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
    
    # Create output filename with _vela suffix
    output_filename = input_path.stem + "_vela" + input_path.suffix
    output_path = output_dir / output_filename
    
    # Build Vela command
    vela_cmd = [
        "vela",
        str(input_path),
        "--accelerator-config", accelerator_config,
        "--output-dir", str(output_dir)
    ]
    
    # Add optional config file
    if config_file:
        vela_cmd.extend(["--config", config_file])
    
    print(f"Compiling {input_path.name} with Vela...")
    print(f"Command: {' '.join(vela_cmd)}")
    
    try:
        # Run Vela compiler
        result = subprocess.run(vela_cmd, capture_output=True, text=True, check=True)
        print(result.stdout)
        
        # Vela outputs a file with _vela suffix automatically
        # Check if the file was created
        expected_output = output_dir / (input_path.stem + "_vela" + input_path.suffix)
        
        if expected_output.exists():
            print(f"\n✓ Successfully compiled!")
            print(f"Output: {expected_output}")
            return str(expected_output)
        else:
            print(f"Warning: Expected output file not found at {expected_output}")
            return None
            
    except subprocess.CalledProcessError as e:
        print(f"Error during compilation:")
        print(e.stderr)
        raise
    except Exception as e:
        print(f"Unexpected error: {str(e)}")
        raise

## 4. Compile a TFLite Model

Now you can compile your TFLite model. Replace the path with your actual model file.

In [ ]:
# Example usage - update with your actual model path
input_model = "C:\\Users\\Jiesheng He\\OneDrive\\Documents\\aifitnessmirror\\output\\models\\combined_fitness_model.tflite"  # Change this to your TFLite model path

# Compile the model
# The output will be saved as model_vela.tflite in the same directory
compiled_model = compile_with_vela(
    input_tflite_path=input_model,
    output_dir=None,  # Use same directory as input, or specify a custom path
    accelerator_config="ethos-u55-128"  # Options: ethos-u55-128, ethos-u55-256, ethos-u65-256, etc.
)

## 5. Batch Compile Multiple Models (Optional)

If you have multiple TFLite models to compile, you can use this batch function.

In [ ]:
def batch_compile_models(model_paths, output_dir=None, accelerator_config="ethos-u55-128"):
    """
    Compile multiple TFLite models with Vela.
    
    Parameters:
    -----------
    model_paths : list of str
        List of paths to TFLite models
    output_dir : str, optional
        Directory to save compiled models
    accelerator_config : str
        Accelerator configuration
    
    Returns:
    --------
    dict : Dictionary mapping input paths to output paths
    """
    results = {}
    
    for model_path in model_paths:
        try:
            print(f"\n{'='*60}")
            output_path = compile_with_vela(model_path, output_dir, accelerator_config=accelerator_config)
            results[model_path] = output_path
        except Exception as e:
            print(f"Failed to compile {model_path}: {str(e)}")
            results[model_path] = None
    
    print(f"\n{'='*60}")
    print(f"Batch compilation complete!")
    print(f"Successfully compiled: {sum(1 for v in results.values() if v is not None)}/{len(model_paths)}")
    
    return results

# Example: Compile multiple models
# model_list = ["model1.tflite", "model2.tflite", "test.tflite"]
# results = batch_compile_models(model_list, output_dir="output/models")